# EDA: Simulated Customer Churn Dataset
**50,000 rows, 14 features, simulated Sri Lanka telecom data**

This notebook explores churn patterns for a simulated telecom base with 13.5% churn. The goal is to find which signals are associated with churn and what the business could test.

**Important:** the data is synthetic, so effects show association in simulated data, not cause. See the README for limitations.

**Hypothesis going in:** a drop in monthly spend would signal churn. **What the data allows:** this cannot be tested here, because no customer's spend falls more than 10% over 3 months.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

df = pd.read_csv("../data/customers.csv")
print(f"Shape: {df.shape}")
print(f"Churn rate: {df.churned.mean():.1%}")
df.head()

## 1. Overall Churn Distribution

**Question:** What is our baseline churn rate and does it vary by product tier?

With 50,000 customers we expect natural variance across tiers — but the question is whether churn is evenly distributed or concentrated in a specific segment. If one tier drives disproportionate churn, retention spend should be redirected there.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df["churned"].value_counts().rename({0:"Retained",1:"Churned"})
axes[0].bar(counts.index, counts.values, color=["#639922","#E24B4A"])
axes[0].set_title("Customer Count by Status")
churn_tier = df.groupby("product_tier")["churned"].mean() * 100
axes[1].bar(churn_tier.index, churn_tier.values, color=["#378ADD","#EF9F27","#639922"])
axes[1].set_title("Churn Rate % by Product Tier")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()
print(churn_tier.round(2))

**Interpretation:** Churn is not evenly distributed across tiers. Standard (15.0%) and Basic (14.3%) churn well above Premium (9.0%), about 40% lower. Tier-specific retention is worth testing, and the reason for the gap cannot be determined from this data.

## 2. Support Ticket Impact

> Customers with 3+ support tickets churn at about 2.8x the rate of customers with 0-2 tickets (about 31% vs 11%).

**Question:** At what point does support friction become a churn signal, and is there still an intervention window?

In [ ]:
ticket_churn = df.groupby("support_tickets")["churned"].mean() * 100
plt.figure(figsize=(10, 5))
plt.bar(ticket_churn.index, ticket_churn.values,
        color=["#E24B4A" if x >= 3 else "#378ADD" for x in ticket_churn.index])
plt.axvline(x=2.5, color="black", linestyle="--", label="Intervention at ticket #2")
plt.xlabel("Number of Support Tickets")
plt.ylabel("Churn Rate %")
plt.title("Churn Rate by Support Ticket Count — RED = intervention zone")
plt.legend()
plt.show()
baseline = ticket_churn.iloc[0]
three_tick = ticket_churn.iloc[3]
print(f"0-ticket churn: {baseline:.1f}%  |  3-ticket churn: {three_tick:.1f}%  |  Multiplier: {three_tick/baseline:.1f}x")

**What this means for the business:** Churn is flat at 0-2 tickets and steps up at ticket 3, so ticket 2 is the point to intervene. This is an association: whether a call at ticket 2 reduces churn must be measured with a holdout test (random Call vs Holdout split, 90-day retention).

Note: tickets 9-12 each have under 50 customers, so those bars are not reliable.

## 3. Inactivity: a step at day 30

**Question:** Is inactivity a gradual drift or a step?

If it is gradual, the timing of a re-engagement message matters less. If it is a step, the message needs to go out before the step.

In [ ]:
df["inactivity_bucket"] = pd.cut(df["last_login_days"],
    bins=[0,7,14,30,60,90,200], labels=["0-7d","8-14d","15-30d","31-60d","61-90d","90d+"])
inactivity_churn = df.groupby("inactivity_bucket")["churned"].mean() * 100
plt.figure(figsize=(10,5))
plt.plot(inactivity_churn.index.astype(str), inactivity_churn.values,
         marker="o", linewidth=2.5, color="#E24B4A", markersize=8)
plt.xlabel("Days Since Last Login")
plt.ylabel("Churn Rate %")
plt.title("Churn Rate by Inactivity Period — send re-engagement email at day 25")
plt.show()

**Interpretation:** Churn is about 10-11% up to day 30 and about 25% from day 31 (day 30 is about 11%, day 31 about 26%). This is a step, not a gradual rise, so re-engagement should happen before day 30.

A perfectly sharp step is typical of simulated data. In real data, expect a smoother rise and test the timing with an experiment.

## 4. Revenue at Risk Summary

**Question:** Which active customers are High Risk right now, and how much monthly revenue do they represent?

High Risk = active customer with 3+ support tickets AND 30+ days inactive. Every customer in this list is still paying.

In [ ]:
high_risk = df[(df.churned == 0) & (df.support_tickets >= 3) & (df.last_login_days > 30)]
print(f"High-risk ACTIVE customers: {len(high_risk):,}")
print(f"Monthly revenue at risk:    LKR {high_risk.monthly_spend.sum():,.0f}")
print(f"\nTop 10 customers on the call list:")
high_risk.sort_values(["support_tickets","monthly_spend"], ascending=False)[["customer_id","product_tier","monthly_spend","support_tickets","last_login_days"]].head(10)

**Business interpretation:** These 783 customers are still active and paying about LKR 51,000 a month in total. This is revenue at risk, not revenue that will certainly be lost.

No cost per retention call or conversion rate is available in this dataset, so no ROI is claimed. The impact of calling them should be measured with a Call vs Holdout test.

This ranked list feeds the Risk page of the Power BI dashboard.